In [1]:
import pickle
import numpy as np
from qkeras.utils import load_qmodel
import hls4ml
from math import ceil, log2
from cnn_base import build_t1_quantizedCNN

2026-08-30 22:26:41.358570: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-30 22:26:41.359464: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-30 22:26:41.376818: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-30 22:26:41.377280: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-30 22:26:41.700229: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [2]:
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
MODEL_PATH = "t1_qkeras_model.h5"
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]
 
 
def accuracy_from_probs(y_true_onehot, y_pred_probs):
    y_true = np.argmax(y_true_onehot, axis=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    return float(np.mean(y_true == y_pred))

 
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)
X_eval, y_eval = X_test, y_test

qmodel = load_qmodel(MODEL_PATH)
#qmodel.summary()

Lendo ./dataset/sat6_test.pkl ...


In [3]:
def accumulator_precision(fan_in, frac_bits=8, guard_bits=1):
    int_bits = ceil(log2(fan_in)) + guard_bits
    total_bits = int_bits + frac_bits
    return f'ap_fixed<{total_bits},{int_bits},AP_RND_CONV,AP_SAT>'

layer_fan_in = {
    'conv1': 4 * 3 * 3,     # 36
    'conv2': 20 * 1 * 1,    # 20
    'fc1':   8 * 16 * 16,   # 2048  <- the layer most at risk of overflow
}

In [4]:
OUTPUT_DIR = "hls4ml_prj_t1_qkeras"
FPGA_PART = "xc7z020clg400-1"
CLOCK_PERIOD_NS = 10
IO_TYPE = "io_stream"
BACKEND = "Vitis"

hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity="name",
    backend='Vitis',
)

hls_config['Model']['Strategy'] = 'Resource'
hls_config['Model']['ReuseFactor'] = 1
hls_config['Model']['Precision'] = 'ap_fixed<8,0,AP_RND_CONV,AP_SAT>'

for layer_name, fan_in in layer_fan_in.items():
    prec = accumulator_precision(fan_in)
    hls_config['LayerName'][layer_name]['accum_t'] = prec
    hls_config['LayerName'][layer_name]['result_t'] = prec
    hls_config['LayerName'][layer_name]['weight_t'] = f'ap_fixed<8,0,AP_RND_CONV,AP_SAT>'

for layer_name in ('relu1', 'relu2'):
    hls_config['LayerName'][layer_name]['Precision'] = f'ap_ufixed<8,0,AP_RND_CONV,AP_SAT>'

hls_config['LayerName']['fc1']['result_t'] = accumulator_precision(
    layer_fan_in['fc1'], frac_bits=8, guard_bits=1 + 1
)

print(hls_config)


{'Model': {'Precision': 'ap_fixed<8,0,AP_RND_CONV,AP_SAT>', 'ReuseFactor': 1, 'Strategy': 'Resource', 'BramFactor': 1000000000, 'TraceOutput': False}, 'LayerName': {'input_layer': {'Trace': False, 'Precision': {'result': 'auto'}}, 'conv1': {'Trace': False, 'Precision': {'result': 'auto', 'weight': 'fixed<8,1,TRN,WRAP,0>', 'bias': 'auto', 'accum': 'auto'}, 'ReuseFactor': 1, 'ParallelizationFactor': 1, 'ConvImplementation': 'LineBuffer', 'accum_t': 'ap_fixed<15,7,AP_RND_CONV,AP_SAT>', 'result_t': 'ap_fixed<15,7,AP_RND_CONV,AP_SAT>', 'weight_t': 'ap_fixed<8,0,AP_RND_CONV,AP_SAT>'}, 'conv1_linear': {'Trace': False, 'Precision': {'result': 'auto', 'table': 'fixed<18,8,TRN,WRAP,0>'}, 'ReuseFactor': 1, 'TableSize': 1024}, 'relu1': {'Trace': False, 'Precision': 'ap_ufixed<8,0,AP_RND_CONV,AP_SAT>', 'ReuseFactor': 1, 'TableSize': 1024}, 'conv2': {'Trace': False, 'Precision': {'result': 'auto', 'weight': 'fixed<8,1,TRN,WRAP,0>', 'bias': 'auto', 'accum': 'auto'}, 'ReuseFactor': 1, 'Parallelization

In [5]:
hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    output_dir=OUTPUT_DIR,
    part=FPGA_PART,
    clock_period=CLOCK_PERIOD_NS,
    io_type=IO_TYPE,
    backend=BACKEND,
)
hls_model.compile()

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/hls4ml/model/graph.py:483: UserWarning: Overriding attribute 'accum_t' of layer 'conv1' (Conv2D):<hls4ml.model.types.NamedType object at 0x7434975e3460> -> <hls4ml.model.types.NamedType object at 0x7434975e1210>
  self.graph[name] = self.make_node(kind, name, layer, inputs, outputs)
/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/hls4ml/model/graph.py:483: UserWarning: Overriding attribute 'accum_t' of layer 'conv2' (Conv2D):<hls4ml.model.types.NamedType object at 0x74349747db10> -> <hls4ml.model.types.NamedType object at 0x74349747ebf0>
  self.graph[name] = self.make_node(kind, name, layer, inputs, outputs)
/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/hls4ml/model/graph.py:483: UserWarning: Overriding attribute 'accum_t' of layer 'fc1' (Dense):<hls4ml.model.types.NamedType object at 0x74349747f9d0> -> <hls4ml.model.types.NamedType object at 0x74349747c430>
 

In [6]:
y_pred_q = qmodel.predict(X_eval)
y_pred_hls = hls_model.predict(X_eval)

acc_q = accuracy_from_probs(y_eval, y_pred_q)
acc_hls = accuracy_from_probs(y_eval, y_pred_hls)

diff = np.abs(y_pred_q - y_pred_hls)

print("\n===== Resultado da comparação (QKeras 8-bit) =====")
print(f"Acurácia QKeras (simulação)  : {acc_q*100:.2f}%")
print(f"Acurácia hls4ml (csim)       : {acc_hls*100:.2f}%")
print(f"Diferença de acurácia        : {abs(acc_q - acc_hls)*100:.2f} pontos percentuais")
print(f"Diferença máxima nas probs   : {diff.max():.6f}")
print(f"Diferença média nas probs    : {diff.mean():.6f}")

2532/2532 [==============================] - 2s 778us/step

===== Resultado da comparação (QKeras 8-bit) =====
Acurácia QKeras (simulação)  : 4.59%
Acurácia hls4ml (csim)       : 4.59%
Diferença de acurácia        : 0.00 pontos percentuais
Diferença máxima nas probs   : nan
Diferença média nas probs    : nan


In [5]:
# --------------------------------------------------------
# 6) (Opcional) Síntese C/RTL de fato -- requer Vivado/Vitis HLS.
# --------------------------------------------------------
# hls_model.build(csim=True, synth=True)
# print(f"\nProjeto HLS gerado em: ./{OUTPUT_DIR}")
# print("Relatórios de síntese em:")
# print(f"  ./{OUTPUT_DIR}/myproject_prj/solution1/syn/report/")